[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_YouBike%E8%88%87%E9%9B%BB%E5%BD%B1%E7%A5%A8%E6%88%BF_%E8%AE%80%E6%87%82%E6%94%BF%E5%BA%9C%E9%96%8B%E6%94%BE%E8%B3%87%E6%96%99.ipynb)

# 🚲 YouBike 與電影票房：用 Python 讀懂真實的政府開放資料

> 🎯 **情境**：你在市政府實習，主管丟給你兩份從**政府資料開放平台**下載的檔案：一份是臺北市 YouBike 每個月被騎了幾次，一份是全台灣每一部電影賣了多少票房。第一份一打開就是亂碼，第二份是一大坨 JSON。今天的任務是：**不用 Excel，只用 Python 內建功能把它們讀懂，回答「YouBike 騎乘量變幾倍了？」和「台灣人最愛看哪部電影？」**

| 素材檔案 | 內容 |
| --- | --- |
| `臺北市YouBike每月使用量.csv` | 臺北市 YouBike 每月騎乘次數（2020 年 11 月起；**Big5 編碼**，直接打開會亂碼！） |
| `全國電影票房_2016至今.json` | 文化部全國電影票房（2016 年至今、7,000 多部電影，JSON 格式） |

| 📡 資料來源（政府資料開放平台） | 提供機關 | 下載日期 |
| --- | --- | --- |
| [YouBike臺北市站位每月使用量](https://data.gov.tw/dataset/135775) | 臺北市政府交通局 | 2026-10-24 |
| [全國電影票房統計數據](https://data.gov.tw/dataset/94224) | 文化部 | 2026-10-24 |

> ✅ 這些都是**真實的政府開放資料**，依「政府資料開放授權條款－第1版」使用。課堂上統一使用上表日期的快照，所以每個人跑出來的數字都一樣。

## 🗺️ 今日任務地圖

| 關卡 | 要回答的問題 |
| --- | --- |
| ⬜ 任務 1 | 打開就是亂碼！先搞定編碼 |
| ⬜ 任務 2 | YouBike 騎乘量，這幾年變幾倍了？ |
| ⬜ 任務 3 | 把結果寫成文字報告 |
| ⬜ 任務 4 | 台灣人最愛看哪部電影？——讀取 JSON |
| ⬜ 任務 5 | 把「2025 年賣座前 10 名」存成 JSON，交給做網站的同學 |

**怎麼用這本 Notebook**：從上到下按 `Shift + Enter` 一格一格執行。每個任務都會先講「老闆想知道什麼」，再給程式，最後用 💡 **看懂結果** 告訴你這個數字代表什麼。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '臺北市YouBike每月使用量.csv',
    '全國電影票房_2016至今.json',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96/%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['臺北市YouBike每月使用量.csv', '全國電影票房_2016至今.json']


## 任務 1：打開就是亂碼！先搞定編碼

**你想知道**：政府的 CSV 為什麼常常打開是亂碼？要怎麼救？

**白話說明**：電腦存中文時要選一本「翻譯字典」（編碼）。台灣很多政府系統還在用舊的 **Big5（cp950）**，新系統用 **UTF-8**。用錯字典讀，Python 會直接報錯 `UnicodeDecodeError`。我們用 `try / except`：先試 UTF-8，失敗就換 cp950。

In [2]:
path = '臺北市YouBike每月使用量.csv'

try:
    with open(path, encoding='utf-8') as f:     # 先用 UTF-8 試試看
        text = f.read()
except UnicodeDecodeError as e:
    print('❌ 用 UTF-8 讀失敗：', e)
    with open(path, encoding='cp950') as f:     # 換成 Big5（cp950）
        text = f.read()
    print('✅ 改用 cp950 成功！')

print(text[:200])                               # 印出前 200 個字確認

❌ 用 UTF-8 讀失敗： 'utf-8' codec can't decode byte 0xa5 in position 0: invalid start byte
✅ 改用 cp950 成功！
民國年,西元年,月份,發布機關名稱,機關代碼,臺北市YouBike每月使用量（次數）
109,2020,11,臺北市政府交通局,379530000H,2730442
109,2020,12,臺北市政府交通局,379530000H,2072168
110,2021,1,臺北市政府交通局,379530000H,2291365
110,2021,2,臺北市政府交通局,379530000H,2208191


In [3]:
# 轉存成 UTF-8：之後所有程式都用 UTF-8 讀，再也不會亂碼
with open('YouBike_utf8.csv', 'w', encoding='utf-8', newline='') as f:
    f.write(text)
print('已另存為 YouBike_utf8.csv')

已另存為 YouBike_utf8.csv


> 💡 **看懂結果**：錯誤訊息說「`0xa5` 這個位元組不是合法的 UTF-8」——這就是 Big5 檔案的典型症狀。以後看到 `UnicodeDecodeError`，第一個念頭就是「換 `cp950` 試試」。
>
> 🤖 **換你問 AI**：「我從政府資料開放平台下載的 CSV 用 UTF-8 打開是亂碼，可能是 Big5。請寫一段 Python 自動嘗試 utf-8、utf-8-sig、cp950，成功後轉存成 UTF-8。」

## 任務 2：YouBike 騎乘量，這幾年變幾倍了？

**你想知道**：每一年總共被騎了幾次？哪一個月最多人騎？

**白話說明**：`csv.DictReader` 會把第一列當欄位名稱，每一列變成一個字典。⚠️ CSV 讀進來的數字都是**文字**，要先用 `int()` 轉成數字才能相加。

In [4]:
import csv

COUNT = '臺北市YouBike每月使用量（次數）'           # 欄位名稱很長，先存成變數

rows = []
with open('YouBike_utf8.csv', encoding='utf-8') as f:
    for row in csv.DictReader(f):                  # 每一列：{'西元年': '2021', '月份': '1', ...}
        row[COUNT] = int(row[COUNT])               # 文字 → 數字
        rows.append(row)

yearly = {}                                        # 用字典累加：{'2021': 27537119, ...}
for r in rows:
    yearly[r['西元年']] = yearly.get(r['西元年'], 0) + r[COUNT]

for year, total in yearly.items():
    months = sum(1 for r in rows if r['西元年'] == year)
    print(f'{year} 年：{total:>12,} 次（{months} 個月的資料）')

best = max(rows, key=lambda r: r[COUNT])           # 找出次數最多的那一列
print(f"\n🏆 史上最多人騎的月份：{best['西元年']} 年 {best['月份']} 月，{best[COUNT]:,} 次")

2020 年：   4,802,610 次（2 個月的資料）
2021 年：  27,537,119 次（12 個月的資料）
2022 年：  34,128,707 次（12 個月的資料）
2023 年：  46,125,019 次（12 個月的資料）
2024 年：  64,963,903 次（12 個月的資料）
2025 年：  79,593,205 次（12 個月的資料）
2026 年：  27,788,438 次（4 個月的資料）

🏆 史上最多人騎的月份：2026 年 3 月，7,751,443 次


> 💡 **看懂結果**：2021 年約 2,750 萬次，2025 年接近 8,000 萬次——**四年變成將近 3 倍**！這段期間正是 YouBike 2.0 全面上路、站點大量增加的時候。
>
> ⚠️ **小心陷阱**：2020 年只有 11、12 兩個月、今年也還沒過完——**資料月份不一樣多，就不能直接比「年總數」**。報告時一定要註明。
>
> 🤖 **換你問 AI**：「這是一份 CSV 的前 5 行（貼上），請用 Python 內建的 csv 模組依年份加總次數，並找出次數最多的月份。」

## 任務 3：把結果寫成文字報告

**你想知道**：主管說「整理成一份文字檔給我，我要貼到週報裡」。

**白話說明**：`open(檔名, 'w')` 是**寫入模式**（檔案存在會被覆蓋）；`f.write()` 不會自動換行，要自己加 `\n`。

In [5]:
with open('YouBike年度報告.txt', 'w', encoding='utf-8') as f:
    f.write('臺北市 YouBike 年度騎乘次數\n')
    f.write('=' * 24 + '\n')
    for year, total in yearly.items():
        f.write(f'{year} 年  {total:>12,} 次\n')
    f.write('\n資料來源：政府資料開放平台（臺北市政府交通局）\n')

print(open('YouBike年度報告.txt', encoding='utf-8').read())

臺北市 YouBike 年度騎乘次數
2020 年     4,802,610 次
2021 年    27,537,119 次
2022 年    34,128,707 次
2023 年    46,125,019 次
2024 年    64,963,903 次
2025 年    79,593,205 次
2026 年    27,788,438 次

資料來源：政府資料開放平台（臺北市政府交通局）



<details><summary>🔄 想看最新資料？點我</summary>

資料每天、每月都在更新。想用今天的數字重跑一次，可以把這段程式貼到新的一格執行：

```python
import urllib.request
url = 'https://data.taipei/api/dataset/d8cefb03-aba2-41ca-9996-d8774313cdc6/resource/8f690548-61bc-4bff-8baa-01d465eb672c/download'
urllib.request.urlretrieve(url, '臺北市YouBike每月使用量.csv')   # 下載後，從任務 1 重新執行即可
```

> ⚠️ 開放資料的欄位偶爾會改名或調整，如果最新資料跑出錯誤，正好練習看懂錯誤訊息、對照欄位名稱修正程式。

</details>

## 任務 4：台灣人最愛看哪部電影？——讀取 JSON

**你想知道**：2016 年到現在，票房最高的電影是哪幾部？

**白話說明**：JSON 就像一層一層的資料夾。先用 `json.load` 讀進來，再看最外層有哪些「鍵」，找到真正的資料在哪一層：這份檔案的電影清單在 `data['List']`。

> ⚠️ 這個 JSON 檔開頭有一個看不見的 BOM 標記，所以編碼要用 `utf-8-sig`（多了 `-sig`），不然會報錯。

In [6]:
import json

with open('全國電影票房_2016至今.json', encoding='utf-8-sig') as f:
    data = json.load(f)                            # JSON → Python 的字典與清單

print('最外層的鍵：', list(data.keys()))
movies = data['List']                              # 真正的電影清單
print('電影數量：', len(movies))
print('第一部電影長這樣：', movies[0])

最外層的鍵： ['Start', 'End', 'List']
電影數量： 7328
第一部電影長這樣： {'Country': '日本', 'Name': '《我在學校旅行中被分到了一個糟糕的小組》高校同學會－大銀幕限定青春重映', 'ReleaseDate': '2026-10-11T00:00:00', 'Issue': '秀泰全球影城股份有限公司', 'Produce': None, 'TheaterCount': 1, 'Tickets': 94, 'TicketChangeRate': 0, 'Amounts': 122200.0, 'TotalTickets': 94, 'TotalAmounts': 122200.0}


In [7]:
top = sorted(movies, key=lambda m: m['TotalAmounts'], reverse=True)[:10]   # 依總票房由高到低，取前 10

print('🎬 2016 年至今台灣票房前 10 名')
for i, m in enumerate(top, 1):
    print(f"{i:>2}. {m['Name']}（{m['Country']}，{m['ReleaseDate'][:4]}）  {m['TotalAmounts'] / 1e8:.2f} 億元")

🎬 2016 年至今台灣票房前 10 名
 1. 復仇者聯盟:終局之戰（美國，2019）  9.10 億元
 2. 劇場版「鬼滅之刃」無限城篇 第一章 猗窩座再襲（日本，2025）  8.57 億元
 3. 陽光女子合唱團（中華民國，2025）  7.75 億元
 4. 動物方城市2 （美國，2025）  7.57 億元
 5. 阿凡達：水之道（美國，2022）  7.50 億元
 6. 捍衛戰士: 獨行俠（美國，2022）  7.41 億元
 7. 玩命關頭8（美國，2017）  6.51 億元
 8. 復仇者聯盟：無限之戰（美國，2018）  6.42 億元
 9. 劇場版「鬼滅之刃」無限列車篇（日本，2020）  6.36 億元
10. 侏羅紀世界:殞落國度（美國，2018）  6.22 億元


> 💡 **看懂結果**：第一名是《復仇者聯盟：終局之戰》（9.1 億），緊追在後的是 2025 年的《鬼滅之刃 無限城篇》。前 10 名幾乎都是美國和日本片——但 2025 年的國片《陽光女子合唱團》也擠進來了！
>
> 🤖 **換你問 AI**：「這是一份 JSON 的前 30 行（貼上），請告訴我電影清單在哪一層，並用 Python 依 TotalAmounts 排序列出前 10 名。」

## 任務 5：把「2025 年賣座前 10 名」存成 JSON，交給做網站的同學

**你想知道**：網站要放一個「年度賣座排行榜」，工程師說給他 JSON 就好。

In [8]:
top2025 = sorted([m for m in movies if m['ReleaseDate'].startswith('2025')],
                 key=lambda m: m['TotalAmounts'], reverse=True)[:10]

result = [{'名次': i, '片名': m['Name'], '國別': m['Country'], '票房(億)': round(m['TotalAmounts'] / 1e8, 2)}
          for i, m in enumerate(top2025, 1)]

with open('2025賣座前10名.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)   # ensure_ascii=False：中文正常顯示；indent=2：排版好讀

print(open('2025賣座前10名.json', encoding='utf-8').read()[:400])

[
  {
    "名次": 1,
    "片名": "劇場版「鬼滅之刃」無限城篇 第一章 猗窩座再襲",
    "國別": "日本",
    "票房(億)": 8.57
  },
  {
    "名次": 2,
    "片名": "陽光女子合唱團",
    "國別": "中華民國",
    "票房(億)": 7.75
  },
  {
    "名次": 3,
    "片名": "動物方城市2 ",
    "國別": "美國",
    "票房(億)": 7.57
  },
  {
    "名次": 4,
    "片名": "F1電影",
    "國別": "美國",
    "票房(億)": 5.74
  },
  {
    "名次": 5,
    "片名": "阿凡達：火與燼",
    "國別": "美國",
    "票房(億)": 4.93
  }


> 💡 **看懂結果**：2025 年前 3 名是日本動畫、國片、美國動畫——三個不同國家的電影輪流稱王。你也可以把年份改成自己出生的那一年看看。

---

## ✍️ 換你動手：練習區

每一題都有三格：**① 半成品程式**（把 `___` 換成正確的程式）→ **② ✅ 檢查格**（自動告訴你對不對）→ **③ 🔑 參考解答**（折疊起來，先自己試）。

> 看到 `NameError: name '___' is not defined` 不用緊張，那代表還有空格沒填。

In [9]:
# @title 🧰 練習檢查小幫手（執行一次就好，不用看懂裡面的程式）
import math
import numpy as np
import pandas as pd


def _same(ans, expected, ordered=False):
    if isinstance(expected, (pd.DataFrame, pd.Series)):
        if type(ans) is not type(expected):
            return False
        try:
            assert_equal = pd.testing.assert_frame_equal if isinstance(expected, pd.DataFrame) else pd.testing.assert_series_equal
            if not ordered:
                ans, expected = ans.sort_index(), expected.sort_index()
            assert_equal(ans, expected, check_names=False, check_dtype=False,
                         check_index_type=False, check_exact=False, rtol=1e-3, atol=0.06,
                         **({'check_column_type': False} if isinstance(expected, pd.DataFrame) else {}))
            return True
        except (AssertionError, TypeError, ValueError):
            return False
    if isinstance(expected, np.ndarray):
        return np.asarray(ans).shape == expected.shape and np.allclose(np.asarray(ans, dtype=float), expected, atol=0.06)
    if isinstance(expected, (int, float, np.number)) and not isinstance(expected, bool):
        try:
            return math.isclose(float(ans), float(expected), rel_tol=1e-3, abs_tol=0.06)
        except (TypeError, ValueError):
            return False
    if isinstance(expected, (list, tuple, set)):
        return list(ans) == list(expected) if not isinstance(expected, set) else set(ans) == expected
    return ans == expected


def check(name, expected, ordered=False):
    """比對學生的變數 name 和正確答案 expected，印出 ✅ 或 ❌；ordered=True 時連排列順序也要一樣。"""
    if name not in globals():
        print(f'⏳ 還找不到變數 {name}：請先把上一格的 ___ 填好並執行')
        return
    ans = globals()[name]
    if _same(ans, expected, ordered):
        print(f'✅ {name} 答對了！')
        return
    print(f'❌ {name} 還不對，再檢查看看：')
    if type(ans) is not type(expected):
        print(f'   型態不同：你的是 {type(ans).__name__}，應該是 {type(expected).__name__}')
    elif hasattr(expected, 'shape') and getattr(ans, 'shape', None) != expected.shape:
        print(f'   大小不同：你的是 {ans.shape}，應該是 {expected.shape}')
    else:
        print('   型態和大小都對了，是數值' + ('或排列順序' if ordered else '') + '不一樣——檢查一下篩選條件或計算方式')


print('🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格')

🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格


### ✍️ 練習 1：2023 年，臺北市 YouBike 總共被騎了幾次？

<details><summary>💡 卡住了？點我看提示</summary>

任務 2 已經算好 `yearly` 字典了，鍵是**文字**的年份，例如 `yearly['2021']`。

</details>

In [ ]:
# ✏️ 練習 1：把 ___ 換成正確的程式，再按執行
rides_2023 = yearly[___]
print(f'2023 年：{rides_2023:,} 次')

In [ ]:
# @title ✅ 執行這格，檢查練習 1
check('rides_2023', sum(r[COUNT] for r in rows if r['西元年'] == '2023'))

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
rides_2023 = yearly['2023']
print(f'2023 年：{rides_2023:,} 次')
```

</details>

### ✍️ 練習 2：國片加油！票房破 1 億元的台灣電影有幾部？

<details><summary>💡 卡住了？點我看提示</summary>

台灣電影的 `Country` 是 `'中華民國'`；1 億可以寫成 `1e8` 或 `100_000_000`。用串列生成式把兩個條件用 `and` 接起來。

</details>

In [ ]:
# ✏️ 練習 2：把 ___ 換成正確的程式，再按執行
tw_hits = [m for m in movies if m['Country'] == ___ and m['TotalAmounts'] >= ___]
print('破億國片：', len(tw_hits), '部')
for m in sorted(tw_hits, key=lambda m: m['TotalAmounts'], reverse=True)[:5]:
    print(' ', m['Name'], round(m['TotalAmounts'] / 1e8, 2), '億')

In [ ]:
# @title ✅ 執行這格，檢查練習 2
check('tw_hits', [m for m in movies if m['Country'] == '中華民國' and m['TotalAmounts'] >= 1e8])

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
tw_hits = [m for m in movies if m['Country'] == '中華民國' and m['TotalAmounts'] >= 1e8]
print('破億國片：', len(tw_hits), '部')
for m in sorted(tw_hits, key=lambda m: m['TotalAmounts'], reverse=True)[:5]:
    print(' ', m['Name'], round(m['TotalAmounts'] / 1e8, 2), '億')
```

</details>

### ✍️ 練習 3：檔案不存在也不要當機：用 `try / except` 印出友善訊息

<details><summary>💡 卡住了？點我看提示</summary>

打開不存在的檔案會丟出 `FileNotFoundError`，把它寫在 `except` 後面。執行後應該只看到「找不到檔案，請確認檔名」。

</details>

In [ ]:
# ✏️ 練習 3：把 ___ 換成正確的程式，再按執行
try:
    with open('新北市YouBike每月使用量.csv', encoding='utf-8') as f:
        print(f.read())
except ___:                                        # 要攔下哪一種錯誤？
    print('找不到檔案，請確認檔名')

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
try:
    with open('新北市YouBike每月使用量.csv', encoding='utf-8') as f:
        print(f.read())
except FileNotFoundError:
    print('找不到檔案，請確認檔名')
```

</details>

---

## 🏆 任務完成！今天你用程式做到了這些事

- ✅ 任務 1：打開就是亂碼！先搞定編碼
- ✅ 任務 2：YouBike 騎乘量，這幾年變幾倍了？
- ✅ 任務 3：把結果寫成文字報告
- ✅ 任務 4：台灣人最愛看哪部電影？——讀取 JSON
- ✅ 任務 5：把「2025 年賣座前 10 名」存成 JSON，交給做網站的同學

> 🎯 **帶走一句話**：程式只是工具，真正有價值的是你從資料裡找到的那句「所以我們應該……」。
> 把上面每個 💡 **看懂結果** 用自己的話寫成一句建議，就是一份可以交給老闆的報告。